# A2 reproduction - Mootoo et al. (2023) GSP + ML on EEG (Table 3, Dementia vs HC)
Run top to bottom on free Colab (CPU is enough). Edit `GITHUB_USER` in the first code cell. Note the real run time in `SETUP.md`.

In [ ]:
GITHUB_USER = 'YOUR-USERNAME'   # <- your GitHub username (your fork must exist)
REPO = 'gsp-alzheimer-detection'
import os
if not os.path.exists(REPO):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!mkdir -p logs results data

In [ ]:
# Record the environment for SETUP.md
!(date; python --version; cat /etc/os-release | head -2; lscpu | head -12; free -h; nvidia-smi 2>&1 | head -3) | tee logs/00_environment.log

In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -2

In [ ]:
# Download data (public OpenNeuro mirror). Keep the log.
!bash scripts/01_download_data.sh data/ds004504
!head -c 600 data/ds004504/dataset_description.json

In [ ]:
# Sanity check of the vectorised total variation
!python tests/check_equivalence.py 2>&1 | grep -v -i warning | tee logs/02a_equivalence.log

In [ ]:
# Feature extraction (checkpoints after every recording). Tip: run once with --limit 3 to time it.
!python scripts/02_extract_features.py --max-bins 1000 2>&1 | grep -v -i warning | tee logs/02_extract.log

In [ ]:
# Reproduce Table 3 with 3 seeds
!python scripts/03_run_models.py --seeds 42 43 44 2>&1 | tee logs/03_models.log

In [ ]:
import pandas as pd
r = pd.read_csv('results/my_results.csv'); display(r.round(3))
p = pd.read_csv('results/per_seed.csv'); display(p[['seed','model','n_test','n_hc_test','accuracy','hc_recall','majority_baseline']].round(3))

In [ ]:
!python make_figure.py
# grayscale check
!pip install -q pdf2image && apt-get -qq install -y poppler-utils > /dev/null
from pdf2image import convert_from_path
from IPython.display import display
img = convert_from_path('fig1.pdf', dpi=200)[0]
display(img); display(img.convert('L'))

In [ ]:
# Save final requirements and download everything
!pip freeze > requirements.txt
!zip -qr results_bundle.zip logs results fig1.pdf requirements.txt
from google.colab import files; files.download('results_bundle.zip')

## After the run
1. Fill `SETUP.md`, `NOTES.md` (section B), the README outcome line and runtime.
2. Commit in 3+ steps, tag `v1.0.0`, and re-test from a fresh clone using only the README.
3. Send the numbers/logs back to Claude to draft the report text, or fill the `.tex` yourself.